In [0]:
STORAGE_ACCOUNT  = "storageaccountfull"
BRONZE_CONTAINER = "bronze"
BRONZE_PATH      = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

CATALOG          = "e-commerce"
SILVER_DB        = "silver"
TARGET_TABLE     = "dim_stores"
NATURAL_KEY      = "store_id"

VALID_STORE_TYPES = ["Retail", "Mall", "Airport", "Online"]

In [0]:
import warnings
warnings.filterwarnings("ignore", message=".*No Partition Defined for Window operation.*")

from pyspark.sql.functions import (
    col, trim, initcap, when, lit,
    current_timestamp, expr,
    row_number, monotonically_increasing_id
)
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import math

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print("Imports ready | Database ensured")

In [0]:
df_bronze = (
    spark.read
    .format("parquet")
    .load(f"{BRONZE_PATH}/stores")
    .withColumn("source_file", col("_metadata.file_path"))
)

print(f"Bronze rows : {df_bronze.count():,}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

In [0]:
df_clean = (
    df_bronze
    .filter(col(NATURAL_KEY).isNotNull())
    .withColumn("store_id",   trim(col("store_id")))
    .withColumn("store_name", trim(col("store_name")))
    .withColumn("city",       trim(initcap(col("city"))))
    .withColumn("country",    trim(initcap(col("country"))))
    .withColumn("store_type", trim(initcap(col("store_type"))))
    .withColumn("store_type",
        when(col("store_type").isin(VALID_STORE_TYPES), col("store_type"))
       .otherwise(lit("Other"))
    )
    .withColumn("channel",
        when(col("store_type") == "Online", lit("Digital"))
       .otherwise(lit("Physical"))
    )
)

print(f"Clean rows : {df_clean.count():,}")

In [0]:
before     = df_clean.count()
df_deduped = df_clean.dropDuplicates([NATURAL_KEY])
print(f"Before : {before:,}  |  After : {df_deduped.count():,}")

In [0]:
if spark.catalog.tableExists(f"{SILVER_DB}.{TARGET_TABLE}"):
    existing_sk = spark.table(f"{SILVER_DB}.{TARGET_TABLE}").select("store_sk", NATURAL_KEY)
    max_sk = existing_sk.agg({"store_sk": "max"}).collect()[0][0] or 0

    df_with_sk = df_deduped.join(existing_sk, on=NATURAL_KEY, how="left")
    df_existing = df_with_sk.filter(col("store_sk").isNotNull())
    df_new = (
        df_with_sk.filter(col("store_sk").isNull()).drop("store_sk")
        .withColumn("_row_id", monotonically_increasing_id())
    )
    window_new = Window.orderBy("_row_id")
    df_new = (
        df_new
        .withColumn("store_sk", (row_number().over(window_new) + max_sk).cast(IntegerType()))
        .drop("_row_id")
    )
    df_with_sk = df_existing.unionByName(df_new)
    print(f"Existing (SK reused) : {df_existing.count():,}")
    print(f"New (SK assigned)    : {df_new.count():,}")

else:
    window_new = Window.orderBy(NATURAL_KEY)
    df_with_sk = df_deduped.withColumn("store_sk", row_number().over(window_new).cast(IntegerType()))
    print(f"First run - SKs 1 -> {df_with_sk.count():,}")

In [0]:
df_dim = (
    df_with_sk
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("updated_at",     current_timestamp())
    .select(
        col("store_sk"),
        col("store_id"),
        col("store_name"),
        col("city"),
        col("country"),
        col("store_type"),
        col("channel"),
        col("ingestion_time"),
        col("source_file"),
        col("updated_at"),
    )
)

print("Final schema:")
df_dim.printSchema()
df_dim.show(5, truncate=False)

In [0]:
if not spark.catalog.tableExists(f"{SILVER_DB}.{TARGET_TABLE}"):
    row_count = df_dim.count()
    num_partitions = max(2, math.ceil(row_count / 10000))
    print(f"{row_count:,} rows -> {num_partitions} partitions")

    df_dim.repartition(num_partitions).write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.{TARGET_TABLE}")
    print(f"{SILVER_DB}.{TARGET_TABLE} created")

else:
    df_dim.createOrReplaceTempView("dim_stores_updates")

    spark.sql(f"""
        MERGE INTO {SILVER_DB}.{TARGET_TABLE}  AS target
        USING dim_stores_updates               AS source
        ON target.store_id = source.store_id

        WHEN MATCHED THEN UPDATE SET
            target.store_name     = source.store_name,
            target.city           = source.city,
            target.country        = source.country,
            target.store_type     = source.store_type,
            target.channel        = source.channel,
            target.updated_at     = source.updated_at

        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"MERGE complete -> {SILVER_DB}.{TARGET_TABLE}")

In [0]:
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {SILVER_DB}.{TARGET_TABLE}").show()

spark.sql(f"""
    SELECT country, store_type, channel, COUNT(*) AS cnt
    FROM {SILVER_DB}.{TARGET_TABLE}
    GROUP BY country, store_type, channel
    ORDER BY country
""").show(30)

spark.sql(f"SELECT * FROM {SILVER_DB}.{TARGET_TABLE} LIMIT 5").show(truncate=False)